# Загрузка моделей

Для анализа тональности используем **Sentiment** модели \
Для английского датасета выбираем SST (единственная доступная) \
Так как датасет содержит русские комменты из твиттера, идеально подошел бы sentiment_twitter (RU), но из-за огранмичений ресурсов (места на диске) выбран rusentiment_bert

In [11]:
from deeppavlov import build_model
ENmodel = build_model('sentiment_sst_conv_bert', download=False, install=False)

Some weights of the model checkpoint at DeepPavlov/bert-base-cased-conversational were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassifi

In [12]:
RUmodel = build_model('rusentiment_bert', download=False, install=False)

Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual

# Подготовка данных

## Импорт

In [96]:
import pandas as pd

ENdf = pd.read_csv("toxic_comments.csv", on_bad_lines='skip',  engine='python')
RUdf = pd.read_csv("rusentitweet_full.csv", on_bad_lines='skip',  engine='python')
#пропускаем неопределенные категории
RUdf = RUdf[RUdf["label"] != "skip"]

In [97]:
ENdf = ENdf.sample(n=20)
ENdf

,Unnamed: 0,text,toxic
21411,21431,dont be sorry... there are plenty of other pag...,0
105196,105293,"""\nWelcome\n\nHello and welcome to Wikipedia! ...",0
62538,62605,I will see if I can identify the various Iraqi...,0
62773,62840,RVD To Go To TNA\n\nVan Dam Has Said In Interv...,0
5538,5538,hey shithead \n\nquit fucking with all the ufc...,1
35637,35679,I think splitting the article was a bad idea a...,0
799,799,Content subsumed into Maneesh page (same entry...,0
54874,54935,"""\n\n Deleted material \n\nThere was some usef...",0
80116,80192,You guys have a lot of balls whining about WP ...,1
43603,43656,Please \n\nEscape this place while you can. Do...,0


In [98]:
RUdf = RUdf.sample(n=20)
RUdf

,Unnamed: 0,text,label,id
11460,11460,Бык уводивший красную тряпку. Просто тряпку.,neutral,1249369185614008320
3653,3653,"Оооо, я вспомнила как меня буллили в школе...\...",negative,1340263540754706432
10597,10597,"@uchiha_mirra Дааа, обожаю🤤",positive,1257976954378207233
4061,4061,"хочу сделать коллаж с инкви-Андерсом, но там в...",neutral,1325797717583532041
4225,4225,"блять, уже завтра мы должны были начать репети...",negative,1260714400438157314
2732,2732,@dm_eliseev Из всей той тусы лишь Пугачиху зна...,neutral,1245163221453062144
2480,2480,@vihoba Берегись автомобиля,neutral,1315952750774161414
9766,9766,когда уже можно суециднуться😔😔😔,negative,1294730969224806401
6522,6522,@bylochkas Ааа...\r\n\r\nПомогает? Для чего это?,neutral,1220688618731327488
7363,7363,Я ещё мой телефон творить в Т9 какую-то хрень ...,negative,1281658850400247810


## Лемматизация (RUdf)

In [104]:
from pymystem3 import Mystem
m = Mystem()
def lemm(text):
    if not text:
        return ""
    lemmas = m.lemmatize(text)
    res = "".join(lemmas).strip()
    return " ".join(res.split())

In [105]:
RUdf['text_lemm'] = RUdf['text'].apply(lambda x: lemm(x))
RUdf

,Unnamed: 0,text,label,id,text_lemm
11460,11460,Бык уводивший красную тряпку. Просто тряпку.,neutral,1249369185614008320,бык уводить красный тряпка. просто тряпка.
3653,3653,"Оооо, я вспомнила как меня буллили в школе...\...",negative,1340263540754706432,"оооо, я вспомнить как я буллили в школа... как..."
10597,10597,"@uchiha_mirra Дааа, обожаю🤤",positive,1257976954378207233,"@uchiha_mirra дааа, обожать🤤"
4061,4061,"хочу сделать коллаж с инкви-Андерсом, но там в...",neutral,1325797717583532041,"хотеть сделать коллаж с инкви-андерс, но там в..."
4225,4225,"блять, уже завтра мы должны были начать репети...",negative,1260714400438157314,"блять, уже завтра мы должный быть начинать реп..."
2732,2732,@dm_eliseev Из всей той тусы лишь Пугачиху зна...,neutral,1245163221453062144,@dm_eliseev из весь тот тус лишь пугачиха знат...
2480,2480,@vihoba Берегись автомобиля,neutral,1315952750774161414,@vihoba беречься автомобиль
9766,9766,когда уже можно суециднуться😔😔😔,negative,1294730969224806401,когда уже можно суециднуться😔😔😔
6522,6522,@bylochkas Ааа...\r\n\r\nПомогает? Для чего это?,neutral,1220688618731327488,@bylochkas ааа... помогать? для что это?
7363,7363,Я ещё мой телефон творить в Т9 какую-то хрень ...,negative,1281658850400247810,я еще мой телефон творить в Т9 какой-то хрень ...


# Классификация

In [110]:
RUdf['byModel'] = RUdf['text_lemm'].apply(lambda x: RUmodel([x])[0])
RUdf
#классификация отличается, но и в изначальном датасете не ахти определен label

,Unnamed: 0,text,label,id,text_lemm,byModel
11460,11460,Бык уводивший красную тряпку. Просто тряпку.,neutral,1249369185614008320,бык уводить красный тряпка. просто тряпка.,negative
3653,3653,"Оооо, я вспомнила как меня буллили в школе...\...",negative,1340263540754706432,"оооо, я вспомнить как я буллили в школа... как...",negative
10597,10597,"@uchiha_mirra Дааа, обожаю🤤",positive,1257976954378207233,"@uchiha_mirra дааа, обожать🤤",positive
4061,4061,"хочу сделать коллаж с инкви-Андерсом, но там в...",neutral,1325797717583532041,"хотеть сделать коллаж с инкви-андерс, но там в...",negative
4225,4225,"блять, уже завтра мы должны были начать репети...",negative,1260714400438157314,"блять, уже завтра мы должный быть начинать реп...",negative
2732,2732,@dm_eliseev Из всей той тусы лишь Пугачиху зна...,neutral,1245163221453062144,@dm_eliseev из весь тот тус лишь пугачиха знат...,negative
2480,2480,@vihoba Берегись автомобиля,neutral,1315952750774161414,@vihoba беречься автомобиль,neutral
9766,9766,когда уже можно суециднуться😔😔😔,negative,1294730969224806401,когда уже можно суециднуться😔😔😔,neutral
6522,6522,@bylochkas Ааа...\r\n\r\nПомогает? Для чего это?,neutral,1220688618731327488,@bylochkas ааа... помогать? для что это?,neutral
7363,7363,Я ещё мой телефон творить в Т9 какую-то хрень ...,negative,1281658850400247810,я еще мой телефон творить в Т9 какой-то хрень ...,neutral


In [112]:
ENdf['byModel'] = ENdf['text'].apply(lambda x: ENmodel([x])[0])
ENdf

,Unnamed: 0,text,toxic,byModel
21411,21431,dont be sorry... there are plenty of other pag...,0,very_negative
105196,105293,"""\nWelcome\n\nHello and welcome to Wikipedia! ...",0,neutral
62538,62605,I will see if I can identify the various Iraqi...,0,neutral
62773,62840,RVD To Go To TNA\n\nVan Dam Has Said In Interv...,0,neutral
5538,5538,hey shithead \n\nquit fucking with all the ufc...,1,very_negative
35637,35679,I think splitting the article was a bad idea a...,0,negative
799,799,Content subsumed into Maneesh page (same entry...,0,neutral
54874,54935,"""\n\n Deleted material \n\nThere was some usef...",0,neutral
80116,80192,You guys have a lot of balls whining about WP ...,1,negative
43603,43656,Please \n\nEscape this place while you can. Do...,0,very_negative


In [116]:
#рассмотрим отличия
print(ENdf.loc[74442]['text'])
#в принципе, негативный подтекст есть

"

Defacing someone else's user page is vandalism and if you do it again you will be blocked. If you want to contest the deletion of this image then leaving increasingly hysterical messages for this user isn't going to do you much good. I suggest you leave a civil, polite message for the administrator who deleted the page (hint: Musamies is not an administrator) or take it to WP:DRV.  "


In [117]:
print(ENdf.loc[43603]['text'])
#ошибка модели, явно не very negative

Please 

Escape this place while you can. Don't end up like me. Having nightmares about this place. I keep on thinking it will completely go away, but it doesn't.


In [119]:
print(ENdf.loc[35637]['text'])
#написано с недовольством, правильно определено

I think splitting the article was a bad idea and awkwardly done; the main article now seems more a victim of amputation than pruning. I agree that it's acceptable to have a large article for now, since this is a 'transitional' phase as events unfold. Splitting off the timeline is enough, IMO.
